# Exploratorio — nada congelado

**Sesión 3 — chequeos previos a las decisiones.** Sin modelos, sin métricas de desempeño, nada escrito en `data/splits/`. Config: `configs/00b_chequeos_previos.yaml`.

In [1]:
import sys, random, json, subprocess, time, os
sys.path.insert(0, "src")
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, yaml
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from foliares.data import grupos_hoja as gh, duplicados as dup, visual
from foliares.data.leaf_grouping import calcular_leaf_id
from foliares.data.taxonomia import clases_plantvillage_de_cultivo, CLASES_PLANTDOC_CANDIDATAS
pd.set_option("display.width", 220, "display.max_rows", 300, "display.max_columns", 50)
cfg = yaml.safe_load(open("configs/00b_chequeos_previos.yaml", encoding="utf-8"))
R = {k: Path(v) for k, v in cfg["rutas"].items()}
FIG = Path(cfg["salida_figuras"]); FIG.mkdir(parents=True, exist_ok=True)
for nombre, ruta in [("plantvillage", "data/raw/plantvillage"), ("plantdoc", "data/raw/plantdoc")]:
    h = subprocess.check_output(["git", "-C", ruta, "log", "-1", "--format=%H"], text=True).strip()
    print(nombre, h, "OK" if h == cfg["commits_fijados"][nombre] else "NO COINCIDE")
CAND_PV = [c for k in ("tomate", "papa", "pimiento") for c in clases_plantvillage_de_cultivo(k)]
MANZANA_PV = ["Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy"]
CAND_PD = {**CLASES_PLANTDOC_CANDIDATAS, "manzana": ["Apple Scab Leaf", "Apple leaf", "Apple rust leaf"]}
CAND_PD_SIN_MANZANA = [c for k, v in CAND_PD.items() if k != "manzana" for c in v]
ren = pd.read_csv(R["pd_renombrados"])
orig = {(r.split, r.clase, r.nombre_local): r.nombre_original for r in ren.itertuples()}
def pd_archivos(split, clase):
    d = (R["pd_train"] if split == "train" else R["pd_test"]) / clase
    return sorted(p for p in d.iterdir() if p.is_file()) if d.is_dir() else []
def pd_nombre_original(split, clase, p):
    return orig.get((split, clase, p.name), p.name)
def corto(p): return str(p).replace("\\", "/").replace("data/raw/plantdoc/", "")

plantvillage 7f7ecc7e1eaca78107e3affe7cb5abd9427e139a OK
plantdoc 5467f6012d78d1c446145d5f582da6096f852ae8 OK


## 1. Grupos de hoja (PlantVillage, carpeta color)
### 1a. Mapeo imagen→hoja y reconciliación con el loader oficial

In [2]:
t0 = time.time()
df = gh.tabla_imagenes(R["pv_color"], R["pv_filtered_leafmaps"], CAND_PV + MANZANA_PV)
lm = json.load(open(R["pv_leaf_map"], encoding="utf-8"))
df["loader"] = [not calcular_leaf_id(a, c, lm).startswith("fallback_") for a, c in zip(df.archivo, df.clase)]
df["cultivo"] = df.clase.str.split("___").str[0]
tom = df[df.cultivo == "Tomato"]
print("Tomate: imágenes", len(tom), "| con grupo (CSV, match exacto):", int(tom.con_grupo.sum()), "| mapeadas por loader oficial:", int(tom.loader.sum()))
rec = df.groupby("clase", sort=False).agg(imagenes=("archivo","size"), csv=("con_grupo","sum"), loader=("loader","sum"))
rec["csv_existe"] = df.groupby("clase", sort=False).csv_existe.first()
rec["filas_csv"] = [len(pd.read_csv(R["pv_filtered_leafmaps"]/f"{c}.csv")) if (R["pv_filtered_leafmaps"]/f"{c}.csv").exists() else None for c in rec.index]
rec["filas_csv_sin_imagen"] = rec.filas_csv - rec.csv
display(rec)
dif = df[df.con_grupo != df.loader]
print("Imágenes donde CSV y loader discrepan:"); display(dif.groupby(["clase","sesion","con_grupo","loader"]).size().rename("n").reset_index())

Tomate: imágenes 18160 | con grupo (CSV, match exacto): 11411 | mapeadas por loader oficial: 11411


,imagenes,csv,loader,csv_existe,filas_csv,filas_csv_sin_imagen
clase,,,,,,
Tomato___Bacterial_spot,2127,2127,2127,True,2128.0,1.0
Tomato___Early_blight,1000,999,999,True,1000.0,1.0
Tomato___Late_blight,1909,920,920,True,920.0,0.0
Tomato___Leaf_Mold,952,952,952,True,952.0,0.0
Tomato___Septoria_leaf_spot,1771,1000,1000,True,1000.0,0.0
Tomato___Spider_mites Two-spotted_spider_mite,1676,1676,1676,True,1676.0,0.0
Tomato___Target_Spot,1404,0,0,True,1404.0,1404.0
Tomato___Tomato_Yellow_Leaf_Curl_Virus,5357,2738,2738,True,2740.0,2.0
Tomato___Tomato_mosaic_virus,373,0,0,False,NaN,NaN


Imágenes donde CSV y loader discrepan:


,clase,sesion,con_grupo,loader,n
0,Apple___Black_rot,JR_FrgE.S,False,True,621


### 1b. Tabla por clase

In [3]:
t1b = gh.resumen_por_clase(df); display(t1b)
t1b_cult = df.groupby("cultivo").agg(imagenes=("archivo","size"), con_grupo=("con_grupo","sum")); t1b_cult["sin_grupo"]=t1b_cult.imagenes-t1b_cult.con_grupo; display(t1b_cult)

,clase,imagenes,con_grupo,sin_grupo,hojas,img_por_hoja_media,img_por_hoja_min,img_por_hoja_max
0,Tomato___Bacterial_spot,2127,2127,0,532,4.00,3.0,4.0
1,Tomato___Early_blight,1000,999,1,250,4.00,3.0,4.0
2,Tomato___Late_blight,1909,920,989,230,4.00,4.0,4.0
3,Tomato___Leaf_Mold,952,952,0,238,4.00,4.0,4.0
4,Tomato___Septoria_leaf_spot,1771,1000,771,250,4.00,4.0,4.0
5,Tomato___Spider_mites Two-spotted_spider_mite,1676,1676,0,419,4.00,4.0,4.0
6,Tomato___Target_Spot,1404,0,1404,0,NaN,NaN,NaN
7,Tomato___Tomato_Yellow_Leaf_Curl_Virus,5357,2738,2619,685,4.00,3.0,4.0
8,Tomato___Tomato_mosaic_virus,373,0,373,0,NaN,NaN,NaN
9,Tomato___healthy,1591,999,592,250,4.00,3.0,4.0


,imagenes,con_grupo,sin_grupo
cultivo,,,
Apple,3171,2550,621
"Pepper,_bell",2475,2473,2
Potato,2152,2152,0
Tomato,18160,11411,6749


### 1c. Clase × sesión de captura × con/sin grupo

In [4]:
df["sesion"] = df.sesion.fillna("(sin parsear)")
t1c = df.groupby(["clase","sesion","con_grupo"]).size().unstack("con_grupo", fill_value=0).rename(columns={True:"con_grupo", False:"sin_grupo"})
for c in ("con_grupo","sin_grupo"):
    if c not in t1c: t1c[c] = 0
display(t1c)
mixtas = t1c[(t1c.con_grupo>0)&(t1c.sin_grupo>0)]
print("Sesiones mixtas (con y sin grupo):"); display(mixtas)
sg = t1c[t1c.con_grupo==0]; cg = t1c[t1c.sin_grupo==0]
print("Sesiones 100% sin grupo:", len(sg), "con", int(sg.sin_grupo.sum()), "imágenes; sesiones 100% con grupo:", len(cg), "con", int(cg.con_grupo.sum()))
print("Imágenes sin grupo fuera de sesiones 100% sin grupo:", int(t1c.sin_grupo.sum() - sg.sin_grupo.sum()))
print("Nombres con decimales:", int(df.decimal.sum()), "| con 'conflicted copy' en el directorio:", int(df.conflicted.sum()))
display(df[df.decimal].groupby(["clase","sesion","con_grupo"]).size())

con_grupo                                                         sin_grupo  con_grupo
clase                                         sesion                                  
Apple___Apple_scab                            FREC_Scab                   0        630
Apple___Black_rot                             JR_FrgE.S                 621          0
Apple___Cedar_apple_rust                      FREC_C.Rust                 0        275
Apple___healthy                               RS_HL                       0       1645
Pepper,_bell___Bacterial_spot                 JR_B.Spot                   0        589
                                              NREC_B.Spot                 0        408
Pepper,_bell___healthy                        JR_HL                       0       1476
                                              Screen Shot                 1          0
                                              bell-pepper-plant-          1          0
Potato___Early_blight                         RS_Early.B                  0       1000
Potato___Late_blight                          RS_LB                       0       1000
Potato___healthy                              RS_HL                       0        152
Tomato___Bacterial_spot                       GCREC_Bact.Sp               0       1756
                                              UF.GRC_BS_Lab Leaf          0        371
Tomato___Early_blight                         RS_Erly.B                   1        999
Tomato___Late_blight                          GHLB                      771          0
                                              GHLB Leaf                  66          0
                                              GHLB_PS Leaf              148          0
                                              GHLB_PS leaf                2          0
                                              GH_HL Leaf                  2          0
                                              RS_Late.B                   0        920
Tomato___Leaf_Mold                            Crnl_L.Mold                 0        952
Tomato___Septoria_leaf_spot                   JR_Sept.L.S               497          0
                                              Keller.St_CG              274          0
                                              Matt.S_CG                   0       1000
Tomato___Spider_mites Two-spotted_spider_mite Com.G_SpM_FL                0       1676
Tomato___Target_Spot                          Com.G_TgS_FL             1404          0
Tomato___Tomato_Yellow_Leaf_Curl_Virus        UF.GRC_YLCV_Lab          2619          0
                                              YLCV_GCREC                  0       1470
                                              YLCV_NREC                   0       1268
Tomato___Tomato_mosaic_virus                  PSU_CG                    373          0
Tomato___healthy                                                          1          0
                                              CG                          1          0
                                              GH_HL Leaf                590          0
                                              RS_HL                       0        999

Sesiones mixtas (con y sin grupo):


,con_grupo,sin_grupo,con_grupo
clase,sesion,,
Tomato___Early_blight,RS_Erly.B,1,999


Sesiones 100% sin grupo: 16 con 7371 imágenes; sesiones 100% con grupo: 18 con 17587
Imágenes sin grupo fuera de sesiones 100% sin grupo: 1
Nombres con decimales: 394 | con 'conflicted copy' en el directorio: 0


clase                 sesion        con_grupo
Tomato___Late_blight  GHLB Leaf     False         41
                      GHLB_PS Leaf  False         81
                      GHLB_PS leaf  False          1
                      GH_HL Leaf    False          1
Tomato___healthy      GH_HL Leaf    False        270
dtype: int64

### 1d. Chequeos de sanidad

In [5]:
todas = {c.name: {gh.sin_uuid(f.name) for f in c.iterdir()} for c in R["pv_color"].iterdir() if c.is_dir()}
cnt = Counter(n for s in todas.values() for n in s)
rep = {n: [c for c, s in todas.items() if n in s] for n, k in cnt.items() if k > 1}
print("Nombres de archivo (sin uuid) presentes en >1 clase, sobre las 38 clases:", len(rep))
rep_cand = {n: v for n, v in rep.items() if any(c in CAND_PV+MANZANA_PV for c in v)}
print("  de ellos, involucrando clases candidatas/manzana:", len(rep_cand), list(rep_cand.items())[:5])
amb = {k: v for k, v in lm.items() if len(v) > 1 and any(s.split(":::")[0] in CAND_PV+MANZANA_PV for s in v)}
print("Claves ambiguas del leaf-map oficial (>1 clase) que tocan candidatas:", len(amb))
if amb:
    display(pd.Series(Counter(tuple(sorted({s.split(":::")[0] for s in v})) for v in amb.values())))
c = df[df.con_grupo]
cruza = c.groupby("hoja").sesion.nunique(); print("Hojas (con grupo) que cruzan sesiones:", int((cruza>1).sum()), "de", len(cruza))
print("Archivos del directorio con 'conflicted'/'copy':", int(df.nombre.str.contains("conflicted|copy").sum()))
for cl in CAND_PV+MANZANA_PV:
    p = R["pv_filtered_leafmaps"]/f"{cl}.csv"
    if p.exists():
        f = pd.read_csv(p, dtype=str)["File Name"]
        k = f[f.str.contains("conflicted|copy")]
        if len(k): print(" CSV", cl, "filas con conflicted/copy:", k.tolist(), "-> ¿existe imagen con ese nombre?", [x in set(df.nombre) for x in k])
print("Early_blight sin grupo:", df[(df.clase=="Tomato___Early_blight")&~df.con_grupo].nombre.tolist())
print("Ejemplos con decimales:", df[df.decimal].nombre.head(3).tolist())
ts = pd.read_csv(R["pv_filtered_leafmaps"]/"Tomato___Target_Spot.csv", dtype=str)
nums_csv = set(ts["File Name"].str.extract(r"(\d+)\.JPG")[0].astype(int))
nums_img = set(df[df.clase=="Tomato___Target_Spot"].num.astype(int))
print("Target_Spot: CSV usa prefijo", ts["File Name"].str.replace(r"\s*\d+.*", "", regex=True).unique().tolist(), "vs imágenes", df[df.clase=="Tomato___Target_Spot"].sesion.unique().tolist(),
      "| números en común (diagnóstico, NO match exacto):", len(nums_csv & nums_img), "de", len(nums_img))

Nombres de archivo (sin uuid) presentes en >1 clase, sobre las 38 clases: 2171
  de ellos, involucrando clases candidatas/manzana: 1377 [('RS_HL 5862.JPG', ['Apple___healthy', 'Soybean___healthy']), ('RS_HL 6192.JPG', ['Apple___healthy', 'Soybean___healthy']), ('RS_HL 6112.JPG', ['Apple___healthy', 'Soybean___healthy']), ('RS_HL 6147.JPG', ['Apple___healthy', 'Soybean___healthy']), ('RS_HL 6108.JPG', ['Apple___healthy', 'Soybean___healthy'])]
Claves ambiguas del leaf-map oficial (>1 clase) que tocan candidatas: 1377


Apple___healthy      Soybean___healthy     nan                  977
Blueberry___healthy  Potato___healthy      Soybean___healthy     20
                     Tomato___healthy      nan                  321
Potato___healthy     Strawberry___healthy  nan                   43
                     Soybean___healthy     nan                   16
dtype: int64

Hojas (con grupo) que cruzan sesiones: 0 de 4166
Archivos del directorio con 'conflicted'/'copy': 0
 CSV Tomato___Early_blight filas con conflicted/copy: ["RS_Erly.B 6341 (Kelsee Baranowski's conflicted copy 2015-10-06).JPG"] -> ¿existe imagen con ese nombre? [False]
Early_blight sin grupo: ['RS_Erly.B 6341.JPG']
Ejemplos con decimales: ['GHLB Leaf 2.1 Day 16.JPG', 'GHLB_PS Leaf 42.1 Day 18.jpg', 'GHLB Leaf 1.1 Day 6.JPG']
Target_Spot: CSV usa prefijo ['Com.G_FL'] vs imágenes ['Com.G_TgS_FL'] | números en común (diagnóstico, NO match exacto): 1404 de 1404


### 1e. Inviabilidad de agrupar por numeración contigua (clases CON grupo)

In [6]:
cand_con = df[~df.cultivo.eq("Apple")]
t1e = gh.contiguidad(cand_con, cfg["contiguidad_tolerancias"]); print("Tomate+papa+pimiento (solo imágenes con grupo):"); display(t1e)
display(gh.contiguidad(df[df.cultivo.eq("Apple")], cfg["contiguidad_tolerancias"]).assign(cultivo="manzana (candidata)"))
rows = []
for cl, g in cand_con[cand_con.con_grupo].groupby("clase"):
    for tol in (3, 10):
        r = gh.contiguidad(g, [tol]).iloc[0]; rows.append({"clase": cl, "tol": tol, "hojas": r.hojas, "partidas_%": r.pct_hojas_partidas, "bloques": r.bloques, "bloque_mayor": r.bloque_mayor})
display(pd.DataFrame(rows))
mos = df[df.clase=="Tomato___Tomato_mosaic_virus"]
print("Tomato mosaic virus:", len(mos), "imágenes; sesión:", mos.sesion.unique().tolist(), "; rango", mos.num.min(), "-", mos.num.max(), "; números distintos", mos.num.nunique(), "; hueco máximo entre consecutivos:", int(np.diff(sorted(mos.num.unique())).max()))
display(gh.contiguidad_clase(mos, cfg["contiguidad_tolerancias"]))

Tomate+papa+pimiento (solo imágenes con grupo):

,tolerancia,bloques,bloque_mayor,hojas,hojas_partidas,pct_hojas_partidas
0,1,2856,631,3732,1245,33.36
1,2,1446,1046,3732,669,17.93
2,3,874,1046,3732,365,9.78
3,5,458,1046,3732,127,3.40
4,10,183,1046,3732,23,0.62


,tolerancia,bloques,bloque_mayor,hojas,hojas_partidas,pct_hojas_partidas,cultivo
0,1,235,611,434,72,16.59,manzana (candidata)
1,2,182,909,434,68,15.67,manzana (candidata)
2,3,156,909,434,62,14.29,manzana (candidata)
3,5,108,909,434,45,10.37,manzana (candidata)
4,10,30,959,434,11,2.53,manzana (candidata)


,clase,tol,hojas,partidas_%,bloques,bloque_mayor
0,"Pepper,_bell___Bacterial_spot",3,175.0,0.00,4.0,309.0
1,"Pepper,_bell___Bacterial_spot",10,175.0,0.00,4.0,309.0
2,"Pepper,_bell___healthy",3,165.0,0.61,7.0,328.0
3,"Pepper,_bell___healthy",10,165.0,0.61,6.0,587.0
4,Potato___Early_blight,3,250.0,42.00,182.0,66.0
5,Potato___Early_blight,10,250.0,0.00,34.0,236.0
6,Potato___Late_blight,3,250.0,5.60,99.0,96.0
7,Potato___Late_blight,10,250.0,0.00,27.0,184.0
8,Potato___healthy,3,38.0,13.16,23.0,20.0
9,Potato___healthy,10,38.0,0.00,5.0,116.0


Tomato mosaic virus: 373 imágenes; sesión: ['PSU_CG'] ; rango 2047.0 - 2420.0 ; números distintos 373 ; hueco máximo entre consecutivos: 2


,tolerancia,bloques,bloque_mayor
0,1,2,206
1,2,1,373
2,3,1,373
3,5,1,373
4,10,1,373


### 1f. No se generó ninguna partición

In [7]:
print("Contenido de data/splits/:", sorted(os.listdir("data/splits")))
print(f"Sección 1: {time.time()-t0:.1f} s")

Contenido de data/splits/: ['.gitkeep']
Sección 1: 1.8 s


## 2. Casi-duplicados (hashing perceptual, solo clases candidatas)

In [8]:
t0 = time.time()
pd_items = [(sp, cl, p) for sp in ("train","test") for cl in CAND_PD_SIN_MANZANA for p in pd_archivos(sp, cl)]
rutas_pd = [p for _,_,p in pd_items]
ok_pd, H8 = dup.hashes_uint64(rutas_pd, con_variantes=True)
assert len(ok_pd) == len(rutas_pd)
meta_pd = pd.DataFrame(pd_items, columns=["split","clase","ruta"]); H0 = H8[:,0]
print("PlantDoc candidatas (sin manzana):", len(meta_pd), meta_pd.groupby("split").size().to_dict(), f"{time.time()-t0:.0f} s")
pv_rutas = [f for c in CAND_PV for f in sorted((R["pv_color"]/c).iterdir())]
ok_pv, HP = dup.hashes_uint64(pv_rutas)
meta_pv = pd.DataFrame({"ruta": ok_pv, "clase": [p.parent.name for p in ok_pv]})
print("PlantVillage candidatas:", len(meta_pv), f"{time.time()-t0:.0f} s")

PlantDoc candidatas (sin manzana): 1100 {'test': 102, 'train': 998} 54 s


PlantVillage candidatas: 22787 72 s


### 2a. Dentro de PlantDoc, misma clase: train↔test y train↔train

In [9]:
def pares_pd(umbral, variantes):
    out = []
    for cl in CAND_PD_SIN_MANZANA:
        idx_tr = np.where((meta_pd.clase==cl)&(meta_pd.split=="train"))[0]; idx_te = np.where((meta_pd.clase==cl)&(meta_pd.split=="test"))[0]
        for tipo, A, B in (("train↔test", idx_tr, idx_te), ("train↔train", idx_tr, idx_tr)):
            if len(A)==0 or len(B)==0: continue
            if variantes:
                D_ = np.min([dup.hamming_matriz(H0[B], H8[A][:,k]) for k in range(8)], axis=0)
            else:
                D_ = dup.hamming_matriz(H0[B], H0[A])
            for ib, ia in zip(*np.where(D_ <= umbral)):
                a, b = A[ia], B[ib]
                if tipo=="train↔train" and a >= b: continue
                out.append({"tipo": tipo, "clase": cl, "dist": int(D_[ib, ia]), "ruta_a": meta_pd.ruta[a], "ruta_b": meta_pd.ruta[b], "split_a": meta_pd.split[a], "split_b": meta_pd.split[b]})
    cols = ["tipo","clase","dist","ruta_a","ruta_b","split_a","split_b"]
    return pd.DataFrame(out, columns=cols).sort_values("dist").reset_index(drop=True)
res2a = {}
for var in (False, True):
    for u in (cfg["duplicados"]["umbral_estricto"], cfg["duplicados"]["umbral_exploratorio"]):
        r = pares_pd(u, var); res2a[(var,u)] = r
        print(f"8 variantes={var} umbral<={u}:", r.groupby("tipo").size().to_dict() if len(r) else "0 pares")
pares_a = res2a[(True, 10)]
display(pares_a.assign(ruta_a=pares_a.ruta_a.map(corto), ruta_b=pares_a.ruta_b.map(corto)).drop(columns=["split_a","split_b"]).head(60))

8 variantes=False umbral<=6: {'train↔test': 4, 'train↔train': 20}
8 variantes=False umbral<=10: {'train↔test': 4, 'train↔train': 20}
8 variantes=True umbral<=6: {'train↔test': 6, 'train↔train': 22}


8 variantes=True umbral<=10: {'train↔test': 6, 'train↔train': 25}


,tipo,clase,dist,ruta_a,ruta_b
0,train↔train,Tomato Early blight leaf,0,train/Tomato Early blight leaf/Early-bligh_t.jpg,train/Tomato Early blight leaf/Early-blight.jpg
1,train↔train,Tomato Early blight leaf,0,train/Tomato Early blight leaf/EBTomato.jpg,train/Tomato Early blight leaf/tomatoblight.jpg
2,train↔train,Tomato Septoria leaf spot,0,train/Tomato Septoria leaf spot/5c77186a4ea326...,train/Tomato Septoria leaf spot/eb56b488e349c2...
3,train↔test,Tomato Septoria leaf spot,0,train/Tomato Septoria leaf spot/early-blight-s...,test/Tomato Septoria leaf spot/early-blight-se...
4,train↔train,Tomato Septoria leaf spot,0,train/Tomato Septoria leaf spot/tomato-septori...,train/Tomato Septoria leaf spot/tomato-septori...
5,train↔train,Tomato Septoria leaf spot,0,train/Tomato Septoria leaf spot/tomato-septori...,train/Tomato Septoria leaf spot/tomato-septori...
6,train↔test,Tomato leaf bacterial spot,0,train/Tomato leaf bacterial spot/bacterial-can...,test/Tomato leaf bacterial spot/07.17.18-Commo...
7,train↔train,Tomato Septoria leaf spot,0,train/Tomato Septoria leaf spot/gray-leaf-spot...,train/Tomato Septoria leaf spot/xgray-leaf-spo...
8,train↔train,Tomato leaf late blight,0,train/Tomato leaf late blight/20150619d.jpg,train/Tomato leaf late blight/IMG_5360.jpg
9,train↔test,Tomato leaf late blight,0,train/Tomato leaf late blight/late_blight1.jpg,test/Tomato leaf late blight/18c.jpg


### 2b. PlantVillage ↔ PlantDoc (PlantDoc con las 8 variantes)

In [10]:
D = np.full((len(H0), len(HP)), 255, dtype=np.uint8); KMIN = np.zeros_like(D)
for k in range(8):
    Dk = dup.hamming_matriz(H8[:,k], HP); m = Dk < D; D[m] = Dk[m]; KMIN[m] = k
D0 = dup.hamming_matriz(H0, HP)
for u in (6, 10):
    print(f"<= {u}: solo identidad {int((D0<=u).sum())} pares | con 8 variantes {int((D<=u).sum())} pares")
def listar(u):
    i, j = np.where(D <= u)
    r = pd.DataFrame({"dist": D[i,j].astype(int), "variante_pd": KMIN[i,j].astype(int), "split_pd": meta_pd.split.values[i], "clase_pd": meta_pd.clase.values[i],
                      "ruta_pd": [corto(meta_pd.ruta[x]) for x in i], "clase_pv": meta_pv.clase.values[j], "ruta_pv": [str(meta_pv.ruta[x]).replace("\\","/").split("color/")[-1] for x in j],
                      "_i": i, "_j": j})
    return r.sort_values(["dist","ruta_pd"]).reset_index(drop=True)
p6 = listar(6); p10 = listar(10)
print("Pares estrictos (<=6):"); display(p6.drop(columns=["_i","_j"]))
print("Exploratorio (<=10):", len(p10), "pares; por split PlantDoc:", p10.groupby("split_pd").size().to_dict())
display(p10.drop(columns=["_i","_j"]).head(60))
cult_pd = lambda c: "Tomato" if c.startswith("Tomato") else ("Potato" if c.startswith("Potato") else "Pepper,_bell")
for nm, pp in (("<=6", p6), ("<=10", p10)):
    mis = pp.clase_pd.map(cult_pd) == pp.clase_pv.str.split("___").str[0]
    print(nm, ": total", len(pp), "| misma especie", int(mis.sum()), "| especie distinta", int((~mis).sum()), "| PlantDoc test:", int((pp.split_pd=="test").sum()))
    print("   imágenes PlantDoc distintas implicadas:", pp._i.nunique(), "| imágenes PlantVillage distintas:", pp._j.nunique())
print("Variante que da el mínimo (<=10):", p10.variante_pd.value_counts().sort_index().to_dict(), "(0 = identidad)")

<= 6: solo identidad 1 pares | con 8 variantes 3 pares
<= 10: solo identidad 43 pares | con 8 variantes 277 pares
Pares estrictos (<=6):


,dist,variante_pd,split_pd,clase_pd,ruta_pd,clase_pv,ruta_pv
0,6,2,test,Potato leaf late blight,test/Potato leaf late blight/irish-blight-symp...,Tomato___Spider_mites Two-spotted_spider_mite,Tomato___Spider_mites Two-spotted_spider_mite/...
1,6,2,train,Potato leaf early blight,train/Potato leaf early blight/irish-blight-sy...,Tomato___Spider_mites Two-spotted_spider_mite,Tomato___Spider_mites Two-spotted_spider_mite/...
2,6,0,train,Tomato leaf late blight,train/Tomato leaf late blight/TomatoLateBlight...,Tomato___Late_blight,Tomato___Late_blight/3de67462-9ae4-455d-8d0d-a...


Exploratorio (<=10): 277 pares; por split PlantDoc: {'test': 92, 'train': 185}


,dist,variante_pd,split_pd,clase_pd,ruta_pd,clase_pv,ruta_pv
0,6,2,test,Potato leaf late blight,test/Potato leaf late blight/irish-blight-symp...,Tomato___Spider_mites Two-spotted_spider_mite,Tomato___Spider_mites Two-spotted_spider_mite/...
1,6,2,train,Potato leaf early blight,train/Potato leaf early blight/irish-blight-sy...,Tomato___Spider_mites Two-spotted_spider_mite,Tomato___Spider_mites Two-spotted_spider_mite/...
2,6,0,train,Tomato leaf late blight,train/Tomato leaf late blight/TomatoLateBlight...,Tomato___Late_blight,Tomato___Late_blight/3de67462-9ae4-455d-8d0d-a...
3,8,6,test,Bell_pepper leaf,test/Bell_pepper leaf/10148582-green-leaf-of-p...,Tomato___Bacterial_spot,Tomato___Bacterial_spot/7f62023d-4d14-42af-b98...
4,8,0,test,Bell_pepper leaf,test/Bell_pepper leaf/10148582-green-leaf-of-p...,Tomato___Leaf_Mold,Tomato___Leaf_Mold/34771f89-a1bc-49e9-8d94-1eb...
5,8,7,test,Bell_pepper leaf,test/Bell_pepper leaf/10148582-green-leaf-of-p...,Potato___Late_blight,Potato___Late_blight/351386f8-c501-45f6-ab0d-e...
6,8,1,test,Potato leaf late blight,test/Potato leaf late blight/1421_0.jpeg_itok=...,Tomato___Tomato_Yellow_Leaf_Curl_Virus,Tomato___Tomato_Yellow_Leaf_Curl_Virus/d86ec73...
7,8,7,test,Potato leaf late blight,test/Potato leaf late blight/1421_0.jpeg_itok=...,Tomato___Tomato_Yellow_Leaf_Curl_Virus,Tomato___Tomato_Yellow_Leaf_Curl_Virus/e873e25...
8,8,7,test,Potato leaf late blight,test/Potato leaf late blight/1421_0.jpeg_itok=...,Potato___Late_blight,Potato___Late_blight/db973ce3-7198-4ae8-910a-b...
9,8,0,test,Potato leaf late blight,test/Potato leaf late blight/1421_0.jpeg_itok=...,Potato___healthy,Potato___healthy/875d6ded-5f01-495c-a945-ad2e1...


<=6 : total 3 | misma especie 1 | especie distinta 2 | PlantDoc test: 1
   imágenes PlantDoc distintas implicadas: 3 | imágenes PlantVillage distintas: 2
<=10 : total 277 | misma especie 138 | especie distinta 139 | PlantDoc test: 92
   imágenes PlantDoc distintas implicadas: 72 | imágenes PlantVillage distintas: 253
Variante que da el mínimo (<=10): {0: 42, 1: 49, 2: 24, 3: 39, 4: 25, 5: 37, 6: 24, 7: 37} (0 = identidad)


### 2c. Grilla de los N pares de menor distancia (elegidos por distancia)

In [11]:
N = cfg["duplicados"]["pares_en_grilla"]
def fig_pares(df_pares, titulo, archivo):
    top = df_pares.head(N)
    fig, axs = plt.subplots(len(top), 2, figsize=(4.2, 2.1*max(1,len(top))), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for k, (_, r) in enumerate(top.iterrows()):
        with Image.open(meta_pd.ruta[r._i]) as im: axs[k,0].imshow(im.convert("RGB"))
        axs[k,0].set_title(f"PlantDoc {r.split_pd} v{r.variante_pd}\n{r.clase_pd[:28]}", fontsize=6)
        with Image.open(meta_pv.ruta[r._j]) as im: axs[k,1].imshow(im.convert("RGB"))
        axs[k,1].set_title(f"PlantVillage d={r.dist}\n{r.clase_pv[:30]}", fontsize=6)
    fig.suptitle(titulo, fontsize=8); fig.tight_layout(); fig.savefig(FIG/archivo, dpi=70); return fig
display(fig_pares(p10, f"PV↔PD: {N} pares de menor distancia", "pares_pv_pd_menor_distancia.png"))
tt = pares_a[pares_a.tipo=="train↔test"]
print("Pares train↔test en PlantDoc (<=10, 8 variantes):", len(tt))
display(tt.assign(ruta_a=tt.ruta_a.map(corto), ruta_b=tt.ruta_b.map(corto)).drop(columns=["split_a","split_b"]).head(30))
def fig_pd(df_pares, titulo, archivo):
    top = df_pares.head(N)
    if len(top)==0: print("sin pares:", titulo); return None
    fig, axs = plt.subplots(len(top), 2, figsize=(4.2, 2.1*len(top)), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for k, (_, r) in enumerate(top.iterrows()):
        for c, (rt, sp) in enumerate(((r.ruta_a, r.split_a), (r.ruta_b, r.split_b))):
            with Image.open(rt) as im: axs[k,c].imshow(im.convert("RGB"))
            axs[k,c].set_title(f"{sp} d={r.dist}\n{r.clase[:30]}", fontsize=6)
    fig.suptitle(titulo, fontsize=8); fig.tight_layout(); fig.savefig(FIG/archivo, dpi=70); return fig
f = fig_pd(tt, "PlantDoc train↔test (menor distancia)", "pares_pd_train_test.png")
if f is not None: display(f)
ttr = pares_a[pares_a.tipo=="train↔train"]; print("Pares train↔train (<=10):", len(ttr))
f = fig_pd(ttr, "PlantDoc train↔train (menor distancia)", "pares_pd_train_train.png")
if f is not None: display(f)

<Figure size 420x2520 with 24 Axes>

Pares train↔test en PlantDoc (<=10, 8 variantes): 6


,tipo,clase,dist,ruta_a,ruta_b
3,train↔test,Tomato Septoria leaf spot,0,train/Tomato Septoria leaf spot/early-blight-s...,test/Tomato Septoria leaf spot/early-blight-se...
6,train↔test,Tomato leaf bacterial spot,0,train/Tomato leaf bacterial spot/bacterial-can...,test/Tomato leaf bacterial spot/07.17.18-Commo...
9,train↔test,Tomato leaf late blight,0,train/Tomato leaf late blight/late_blight1.jpg,test/Tomato leaf late blight/18c.jpg
13,train↔test,Tomato leaf yellow virus,0,train/Tomato leaf yellow virus/tylcv-seminar-1...,test/Tomato leaf yellow virus/tylcv-seminar-1-...
21,train↔test,Potato leaf early blight,2,train/Potato leaf early blight/potato-early-bl...,test/Potato leaf early blight/potato-early-bli...
22,train↔test,Potato leaf early blight,2,train/Potato leaf early blight/early-blight-al...,test/Potato leaf early blight/early-blight-alt...


<Figure size 420x1260 with 12 Axes>

Pares train↔train (<=10): 25


<Figure size 420x2520 with 24 Axes>

### 2d. Par ya detectado: `TomatoLateBlightTop.jpg`

In [12]:
m = p6[p6.ruta_pd.str.contains("TomatoLateBlightTop")]
print(m.drop(columns=["_i","_j"]).to_string())
r = m.iloc[0]
a = meta_pd.ruta[r._i]; b = meta_pv.ruta[r._j]
with Image.open(a) as ia, Image.open(b) as ib:
    ia = ia.convert("RGB"); ib = ib.convert("RGB")
    print("PlantDoc:", ia.size, "| PlantVillage:", ib.size)
    fig, axs = plt.subplots(1, 3, figsize=(9, 3.2))
    axs[0].imshow(ia); axs[0].set_title("PlantDoc (original)", fontsize=8)
    axs[1].imshow(dup.variantes_d4(ia)[int(r.variante_pd)]); axs[1].set_title(f"PlantDoc variante {int(r.variante_pd)}", fontsize=8)
    axs[2].imshow(ib); axs[2].set_title("PlantVillage", fontsize=8)
    for ax in axs: ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG/"par_TomatoLateBlightTop.png", dpi=80); display(fig)
print(f"Sección 2: {time.time()-t0:.0f} s")

   dist  variante_pd split_pd                 clase_pd                                                ruta_pd              clase_pv                                                                         ruta_pv
2     6            0    train  Tomato leaf late blight  train/Tomato leaf late blight/TomatoLateBlightTop.jpg  Tomato___Late_blight  Tomato___Late_blight/3de67462-9ae4-455d-8d0d-a625c92c7120___RS_Late.B 5148.JPG
PlantDoc: (3111, 2364) | PlantVillage: (256, 256)


<Figure size 900x320 with 3 Axes>

Sección 2: 89 s


## 3. Muestra visual (semilla 42)
Una sola instancia `random.Random(42)`; rutas ordenadas con `sorted()` antes de muestrear; orden de llamadas fijo (el orden del notebook).

In [13]:
rng = random.Random(cfg["semilla_muestras"])
def muestra(lista, n): return rng.sample(sorted(lista), min(n, len(lista)))
ENF = {"Tomato leaf": [c for c in CAND_PD["tomate"] if c != "Tomato leaf"], "Bell_pepper leaf": ["Bell_pepper leaf spot"], "Apple leaf": ["Apple Scab Leaf", "Apple rust leaf"]}
log_3a = {}
for sana, enfs in ENF.items():
    enf = rng.choice(sorted(enfs))
    s = muestra(pd_archivos("train", sana), cfg["muestras"]["pd_sanas_train"]); e = muestra(pd_archivos("train", enf), cfg["muestras"]["pd_enfermedad_train"])
    log_3a[sana] = {"enfermedad_elegida": enf, "n_sana": len(s), "n_enf": len(e)}
    visual.grilla([(p, pd_nombre_original("train", sana, p)[:22]) for p in s], ncols=8, titulo=f"PlantDoc TRAIN — '{sana}' (n={len(s)})", guardar=FIG/f"pd_train_{sana.replace(' ','_')}.png"); display(plt.gcf())
    visual.grilla([(p, pd_nombre_original("train", enf, p)[:22]) for p in e], ncols=8, titulo=f"PlantDoc TRAIN — contraste '{enf}' (n={len(e)})", guardar=FIG/f"pd_train_contraste_{enf.replace(' ','_')}.png"); display(plt.gcf())
print(log_3a)

<Figure size 1280x555 with 24 Axes>

<Figure size 1280x370 with 16 Axes>

<Figure size 1280x555 with 24 Axes>

<Figure size 1280x370 with 16 Axes>

<Figure size 1280x555 with 24 Axes>

<Figure size 1280x370 with 16 Axes>

{'Tomato leaf': {'enfermedad_elegida': 'Tomato Septoria leaf spot', 'n_sana': 24, 'n_enf': 12}, 'Bell_pepper leaf': {'enfermedad_elegida': 'Bell_pepper leaf spot', 'n_sana': 24, 'n_enf': 12}, 'Apple leaf': {'enfermedad_elegida': 'Apple rust leaf', 'n_sana': 24, 'n_enf': 12}}


In [14]:
log_3b = []
for cultivo, clases in (("tomate", clases_plantvillage_de_cultivo("tomate")), ("papa", clases_plantvillage_de_cultivo("papa")), ("pimiento", clases_plantvillage_de_cultivo("pimiento")), ("manzana", MANZANA_PV)):
    items = []
    for cl in clases:
        g = df[df.clase == cl]; ses = sorted(g.sesion.unique()); cg_ = g.set_index("archivo").con_grupo
        grupos = [(None, g)] if len(ses) == 1 else [(s, g[g.sesion == s]) for s in ses]
        for s, gs in grupos:
            n = cfg["muestras"]["pv_por_clase"] if s is None else cfg["muestras"]["pv_por_sesion"]
            ps = muestra([R["pv_color"]/cl/a for a in gs.archivo], n); ss = ses[0] if s is None else s
            items += [(p, f"{cl.split('___')[1][:14]}|{ss[:10]}|{'G' if cg_[p.name] else 'sinG'}") for p in ps]; log_3b.append((cl, ss, len(ps)))
    visual.grilla(items, ncols=8, lado=1.5, titulo=f"PlantVillage color — {cultivo} (G = con grupo de hoja, sinG = sin grupo)", guardar=FIG/f"pv_{cultivo}.png"); display(plt.gcf())
print(len(log_3b), "celdas clase/sesión muestreadas")

<Figure size 1200x2275 with 104 Axes>

<Figure size 1200x525 with 24 Axes>

<Figure size 1200x350 with 16 Axes>

<Figure size 1200x700 with 32 Axes>

35 celdas clase/sesión muestreadas


In [15]:
items_test = {}
for cultivo, clases in CAND_PD.items():
    items_test[cultivo] = []
    for cl in clases:
        items_test[cultivo] += [(p, cl[:26]) for p in muestra(pd_archivos("test", cl), cfg["muestras"]["pd_test_por_clase"])]
N_TEST_VISTAS = sum(len(v) for v in items_test.values())
for cultivo, sub in items_test.items():
    visual.grilla(sub, ncols=6, lado=1.8, titulo=f"PlantDoc TEST (solo verificar etiquetas/contenido) — {cultivo}", guardar=FIG/f"pd_test_{cultivo}.png"); display(plt.gcf())
print("Imágenes de test inspeccionadas visualmente: n =", N_TEST_VISTAS)

<Figure size 1080x820 with 24 Axes>

<Figure size 1080x205 with 6 Axes>

<Figure size 1080x205 with 6 Axes>

<Figure size 1080x410 with 12 Axes>

Imágenes de test inspeccionadas visualmente: n = 45


### 3d. Resolución y relación de aspecto (sin modelo)
PlantDoc: «dev» = carpeta `train` oficial (la partición dev/test propia NO está decidida) vs. `test` oficial.

In [16]:
def tam(p):
    with Image.open(p) as im: return im.size
filas = []
for sp in ("train","test"):
    for cultivo, clases in CAND_PD.items():
        for cl in clases:
            for p in pd_archivos(sp, cl):
                w, h = tam(p); filas.append({"split": "dev(train)" if sp=="train" else "test", "cultivo": cultivo, "clase": cl, "w": w, "h": h})
res = pd.DataFrame(filas); res["aspecto"] = res.w/res.h; res["lado_min"] = res[["w","h"]].min(axis=1)
def agg(g): return pd.Series({"n": len(g), "w_med": g.w.median(), "h_med": g.h.median(), "lado_min_p10": g.lado_min.quantile(.1), "lado_min_min": g.lado_min.min(), "lado_min_<224_%": round(100*(g.lado_min<224).mean(),1),
                              "aspecto_med": round(g.aspecto.median(),2), "aspecto_p10": round(g.aspecto.quantile(.1),2), "aspecto_p90": round(g.aspecto.quantile(.9),2), "retrato_%": round(100*(g.aspecto<1).mean(),1), "res_distintas": g[["w","h"]].drop_duplicates().shape[0]})
t3d = res.groupby(["clase","split"]).apply(agg).reset_index(); display(t3d)
display(res.groupby(["cultivo","split"]).apply(agg).reset_index())
pvs = Counter(tam(f) for f in pv_rutas); print("PlantVillage candidatas — resoluciones distintas:", dict(pvs))
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
for sp, c in (("dev(train)","tab:blue"),("test","tab:orange")):
    s = res[res.split==sp]; axs[0].hist(np.log2(s.lado_min), bins=30, alpha=.6, label=sp, color=c); axs[1].hist(s.aspecto.clip(0,3), bins=30, alpha=.6, label=sp, color=c)
axs[0].axvline(np.log2(224), color="k", ls="--"); axs[0].set_xlabel("log2(lado menor, px)"); axs[1].set_xlabel("relación de aspecto w/h (recortada a 3)"); axs[0].legend(); fig.tight_layout(); fig.savefig(FIG/"resolucion_aspecto_pd.png", dpi=70); display(fig)
print("Registro para accesos_test.md: n =", N_TEST_VISTAS, "imágenes de test vistas;", int((res.split=="test").sum()), "imágenes de test con estadísticas de resolución (solo cabeceras)")
print("data/splits/ intacto:", sorted(os.listdir("data/splits")))

,clase,split,n,w_med,h_med,lado_min_p10,lado_min_min,lado_min_<224_%,aspecto_med,aspecto_p10,aspecto_p90,retrato_%,res_distintas
0,Apple Scab Leaf,dev(train),83.0,612.0,459.0,200.4,119.0,13.3,1.33,0.75,1.53,20.5,79.0
1,Apple Scab Leaf,test,10.0,493.0,383.5,223.6,85.0,10.0,1.32,1.14,1.43,10.0,10.0
2,Apple leaf,dev(train),82.0,927.5,782.5,380.0,300.0,0.0,1.08,0.75,1.40,34.1,74.0
3,Apple leaf,test,9.0,1300.0,1174.0,672.4,594.0,0.0,1.16,0.56,1.44,33.3,6.0
4,Apple rust leaf,dev(train),79.0,736.0,600.0,246.6,150.0,6.3,1.33,0.75,1.51,20.3,69.0
5,Apple rust leaf,test,10.0,537.5,415.5,184.7,155.0,20.0,1.33,0.92,1.78,20.0,9.0
6,Bell_pepper leaf,dev(train),53.0,800.0,756.0,330.4,230.0,0.0,1.33,0.75,1.50,35.8,43.0
7,Bell_pepper leaf,test,8.0,720.0,600.0,384.7,300.0,0.0,1.33,0.69,1.48,37.5,7.0
8,Bell_pepper leaf spot,dev(train),62.0,800.0,629.5,300.0,180.0,6.5,1.33,0.75,1.50,22.6,41.0
9,Bell_pepper leaf spot,test,9.0,638.0,479.0,200.4,166.0,22.2,1.33,1.22,1.54,11.1,9.0


,cultivo,split,n,w_med,h_med,lado_min_p10,lado_min_min,lado_min_<224_%,aspecto_med,aspecto_p10,aspecto_p90,retrato_%,res_distintas
0,manzana,dev(train),244.0,752.5,600.0,260.8,119.0,6.6,1.30,0.75,1.50,25.0,209.0
1,manzana,test,29.0,768.0,514.0,228.8,85.0,10.3,1.33,0.59,1.56,20.7,24.0
2,papa,dev(train),206.0,750.0,532.5,295.5,195.0,1.9,1.33,0.85,1.68,15.0,172.0
3,papa,test,16.0,564.0,490.0,283.0,150.0,6.2,1.33,0.76,1.48,18.8,16.0
4,pimiento,dev(train),115.0,800.0,667.0,300.0,180.0,3.5,1.33,0.75,1.50,28.7,76.0
5,pimiento,test,17.0,640.0,582.0,263.6,166.0,11.8,1.33,0.75,1.50,23.5,16.0
6,tomate,dev(train),677.0,640.0,561.0,236.0,85.0,8.1,1.33,0.75,1.51,24.2,439.0
7,tomate,test,69.0,638.0,600.0,250.4,69.0,7.2,1.33,0.84,1.59,14.5,61.0


PlantVillage candidatas — resoluciones distintas: {(256, 256): 22787}


<Figure size 900x320 with 2 Axes>

Registro para accesos_test.md: n = 45 imágenes de test vistas; 131 imágenes de test con estadísticas de resolución (solo cabeceras)
data/splits/ intacto: ['.gitkeep']
